# Minimal RAG Solution — HPE Customer Engineering

This notebook demonstrates a complete Retrieval-Augmented Generation pipeline that runs entirely on CPU. We walk through each of the five required components: Knowledge Base, Semantic Layer, Retrieval System, Augmentation, and Generation. Each section is self-contained and runnable. By the end, you will see the full pipeline answer questions using only the information from our document collection.

In [ ]:
# ============================================================
# Install Dependencies
# ============================================================
# Run this cell once. If packages are already installed, it completes in seconds.
# After first install, you may need to restart the kernel.

import subprocess
import sys

def install(package):
    """Install a package using pip from within the notebook."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

packages = [
    "chromadb>=0.5.0",
    "sentence-transformers>=3.0.0",
    "transformers>=4.44.0",
    "torch>=2.1.0",
    "accelerate>=0.27.0",
    "langchain>=0.3.0",
    "langchain-community>=0.3.0",
    "pypdf>=4.0.0",
]

for pkg in packages:
    install(pkg)

print("All dependencies installed.")

## 1. Environment Setup & Configuration

This section installs dependencies, imports libraries, and defines global runtime parameters. It prepares the execution environment shared across all RAG components.

In [ ]:
# ============================================================
# Imports and Configuration
# ============================================================
# Central configuration — every parameter the system uses is defined here.
# This makes the system easy to tune and every choice visible.

import os
import glob
import time
from typing import List

# --- Core libraries ---
import chromadb                              # Vector database
from sentence_transformers import SentenceTransformer  # Embedding model
from transformers import AutoModelForCausalLM, AutoTokenizer  # LLM
import torch

# --- Document processing ---
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ========================
# CONFIGURATION
# ========================
# Embedding model: lightweight, fast on CPU, 384-dimensional vectors
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Generation model: 1.7B params, Apache 2.0, strong instruction-following
LLM_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# ChromaDB persistent storage path
CHROMA_PERSIST_DIR = "./chroma_store"

# Document source directory
DATA_DIR = "./data"

# Collection name in ChromaDB
COLLECTION_NAME = "knowledge_base"

# Chunking parameters
CHUNK_SIZE = 512       # Characters per chunk (~100-130 tokens)
CHUNK_OVERLAP = 64     # Overlap between chunks (prevents info loss at boundaries)

# Retrieval parameters
TOP_K = 5              # Number of chunks to retrieve per query

# Generation parameters
MAX_NEW_TOKENS = 300   # Maximum tokens the LLM generates
TEMPERATURE = 0.3      # Low temperature = more deterministic, factual answers
TOP_P = 0.9            # Nucleus sampling threshold

print("Configuration loaded:")
print(f"  Embedding model : {EMBEDDING_MODEL_NAME}")
print(f"  LLM model       : {LLM_MODEL_NAME}")
print(f"  ChromaDB path   : {CHROMA_PERSIST_DIR}")
print(f"  Data directory  : {DATA_DIR}")
print(f"  Chunk size      : {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP} chars")
print(f"  Retrieval top-K : {TOP_K}")
print(f"  Device          : CPU (no GPU required)")

In [ ]:
# ============================================================
# Create Sample Dataset
# ============================================================
# If the data/ directory is empty, we create sample documents.
# In a real scenario, you would place your own PDFs or text files here.
#
# Dataset topic: Cloud Computing & AI Infrastructure
# (chosen because it's relevant to HPE's business and easy to discuss)

SAMPLE_DOCUMENTS = {
    "cloud_computing_fundamentals.txt": """Cloud computing is the delivery of computing services over the internet. These services include servers, storage, databases, networking, software, analytics, and intelligence. Cloud computing offers faster innovation, flexible resources, and economies of scale. Organizations typically pay only for the cloud services they use, helping them lower operating costs, run infrastructure more efficiently, and scale as business needs change.

The three primary service models in cloud computing are Infrastructure as a Service (IaaS), Platform as a Service (PaaS), and Software as a Service (SaaS). IaaS provides fundamental compute, network, and storage resources on demand, on a pay-as-you-go basis. Examples include virtual machines, storage accounts, and virtual networks. PaaS provides an environment for building, testing, and deploying software applications without the complexity of managing the underlying infrastructure. SaaS delivers software applications over the internet on a subscription basis, with the provider managing the infrastructure, middleware, app software, and security.

Deployment models include public cloud, private cloud, and hybrid cloud. Public clouds are owned and operated by third-party providers who deliver resources over the internet. Private clouds are used exclusively by a single organization, offering greater control and security. Hybrid cloud combines public and private clouds, allowing data and applications to move between them for greater flexibility and optimization.""",

    "edge_computing_overview.txt": """Edge computing is a distributed computing paradigm that brings computation and data storage closer to the sources of data. This approach improves response times and saves bandwidth compared to centralized cloud-only architectures. Edge computing processes data at or near the point of generation rather than relying on a centralized data center that may be hundreds of miles away.

The key benefits of edge computing include reduced latency, bandwidth optimization, enhanced privacy and security, and improved reliability. By processing data closer to where it is generated, edge computing minimizes the round-trip time required for data to travel to a centralized data center and back. This is particularly critical for real-time applications such as autonomous vehicles, industrial IoT, augmented reality, and remote healthcare monitoring.

Edge computing and cloud computing are complementary technologies. Edge computing handles time-sensitive data processing locally, while cloud computing processes data that is not time-driven. In distributed architectures, edge computing manages localized short-term data, while cloud computing manages long-term data storage, batch analytics, and more complex processing tasks that benefit from centralized resources.

Common edge computing use cases include smart manufacturing where sensors on factory floors need millisecond response times, retail analytics where in-store cameras process customer behavior locally, telecommunications where 5G networks rely on edge nodes for low-latency services, and healthcare where medical devices process patient data locally for immediate clinical decisions.""",

    "ai_infrastructure_and_rag.txt": """AI infrastructure refers to the hardware, software, and services required to develop, train, deploy, and manage artificial intelligence and machine learning workloads. This includes GPUs, TPUs, and specialized AI accelerators for training and inference, as well as high-capacity storage systems designed for the massive datasets used in AI development.

Modern AI infrastructure typically involves high-performance computing clusters with GPU-accelerated servers, high-speed networking such as InfiniBand or RoCE, parallel file systems, and container orchestration platforms like Kubernetes. The software stack includes deep learning frameworks like PyTorch and TensorFlow, model serving platforms, MLOps tools for lifecycle management, and monitoring systems for performance and resource utilization.

The rise of large language models and generative AI has dramatically increased demand for AI infrastructure. Training a frontier large language model can require thousands of GPUs running continuously for weeks or months, consuming significant energy and generating substantial heat. This has driven interest in energy-efficient AI infrastructure, including advanced liquid cooling systems, renewable energy-powered data centers, and more efficient chip architectures.

Retrieval-Augmented Generation, commonly known as RAG, is an architecture pattern that combines the generative capabilities of large language models with external information retrieval systems. A RAG system retrieves relevant documents or passages from a knowledge base and uses them as context for the language model to generate more accurate, grounded responses. This approach reduces hallucinations by anchoring the model's output in verifiable, retrieved information rather than relying solely on the model's parametric knowledge.

The core components of a RAG system are a knowledge base for storing documents, an embedding model for creating semantic representations, a vector database for efficient similarity search, a retrieval mechanism to find relevant information, and a generative model to produce the final response. RAG has become the standard approach for building question-answering systems, chatbots, and knowledge management tools that need to stay current with changing information.""",

    "hybrid_cloud_management.txt": """Hybrid cloud management involves the orchestration, monitoring, and optimization of computing resources that span both private infrastructure and public cloud services. Organizations adopt hybrid cloud strategies to balance cost efficiency with security requirements, placing sensitive workloads on private infrastructure while leveraging public cloud for variable or burst capacity.

Key challenges in hybrid cloud management include ensuring consistent security policies across environments, managing data movement between on-premises and cloud systems, maintaining visibility into resource usage and costs, and avoiding vendor lock-in. Effective hybrid cloud management platforms provide a unified control plane that abstracts the differences between environments and presents a consistent operational experience.

Workload placement decisions in hybrid cloud environments consider factors such as data sensitivity, regulatory compliance requirements, latency needs, cost optimization, and scalability demands. Some workloads, like those processing personally identifiable information, may be restricted to on-premises infrastructure by regulation. Other workloads, like development and testing environments, benefit from the elasticity and cost model of public cloud.

Modern hybrid cloud platforms often incorporate AI-driven operations, sometimes called AIOps, to automate routine management tasks, predict capacity needs, detect anomalies, and optimize resource allocation. These platforms use machine learning to analyze telemetry data from across the hybrid estate and provide actionable recommendations for improving performance, reliability, and cost efficiency."""
}

os.makedirs(DATA_DIR, exist_ok=True)

existing_files = glob.glob(os.path.join(DATA_DIR, "*.txt")) + glob.glob(os.path.join(DATA_DIR, "*.pdf"))

if not existing_files:
    print("No documents found in ./data/ — creating sample dataset...\n")
    for filename, content in SAMPLE_DOCUMENTS.items():
        filepath = os.path.join(DATA_DIR, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content.strip())
        print(f"  Created: {filepath} ({len(content.strip())} chars)")
    print(f"\nSample dataset ready: {len(SAMPLE_DOCUMENTS)} documents")
else:
    print(f"Found {len(existing_files)} existing document(s) in ./data/")
    for f in existing_files:
        print(f"  {f}")

## 2. Knowledge Base — Document Loading & Chunking

This section ingests source files and splits them into overlapping chunks for indexing. It implements the Knowledge Base component by preparing structured context units for retrieval.

In [ ]:
# ============================================================
# Load and Chunk Documents
# ============================================================
# Step 1 of the indexing pipeline:
#   Documents → Load → Chunk → (next cell: Embed → Store)
#
# We use LangChain's document loaders for consistent handling of
# different file formats, and RecursiveCharacterTextSplitter for
# intelligent chunking that respects paragraph/sentence boundaries.

def load_documents(data_dir: str) -> list:
    """
    Load all supported documents from the data directory.

    Supports: .pdf (via PyPDF) and .txt (via TextLoader)
    Returns: list of LangChain Document objects with page_content and metadata.
    """
    documents = []

    for pdf_path in sorted(glob.glob(os.path.join(data_dir, "*.pdf"))):
        print(f"  Loading PDF:  {os.path.basename(pdf_path)}")
        loader = PyPDFLoader(pdf_path)
        documents.extend(loader.load())

    for txt_path in sorted(glob.glob(os.path.join(data_dir, "*.txt"))):
        print(f"  Loading TXT:  {os.path.basename(txt_path)}")
        loader = TextLoader(txt_path, encoding="utf-8")
        documents.extend(loader.load())

    return documents


def chunk_documents(documents: list, chunk_size: int, chunk_overlap: int) -> list:
    """
    Split documents into overlapping chunks.

    RecursiveCharacterTextSplitter tries these separators in order:
      1. Double newline (paragraph boundary)
      2. Single newline (line boundary)
      3. Period + space (sentence boundary)
      4. Space (word boundary)
      5. Empty string (character boundary — last resort)

    This hierarchy keeps chunks as semantically coherent as possible.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    return splitter.split_documents(documents)


# --- Execute ---
print("STEP 1: Loading documents...")
raw_documents = load_documents(DATA_DIR)
print(f"  → Loaded {len(raw_documents)} document segment(s)\n")

print("STEP 2: Chunking documents...")
chunks = chunk_documents(raw_documents, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"  → Created {len(chunks)} chunks")
print(f"  → Average chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars\n")

# --- Preview ---
print("Sample chunk (index 0):")
print(f"  Source:  {chunks[0].metadata.get('source', 'N/A')}")
print(f"  Length:  {len(chunks[0].page_content)} chars")
print(f"  Content: {chunks[0].page_content[:300]}...")

## 3. Semantic Layer — Embedding & Vector Store

This section encodes chunks into dense vectors and indexes them in ChromaDB for similarity search. It implements the Semantic Layer that makes meaning-based retrieval possible.

In [ ]:
# ============================================================
# Initialize Embedding Model & Build Vector Store
# ============================================================
# This cell:
#   1. Loads the embedding model (downloads ~80MB on first run)
#   2. Embeds all document chunks
#   3. Stores everything in ChromaDB with persistent storage
#
# After this cell, the knowledge base is ready for retrieval.

# --- Load embedding model ---
print("Loading embedding model...")
t0 = time.time()
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"  Model loaded in {time.time() - t0:.1f}s")
print(f"  Embedding dimension: {embedder.get_embedding_dimension()}")
print(f"  Model size: ~80 MB\n")

# --- Initialize ChromaDB ---
print("Initializing ChromaDB...")
chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

# Delete and recreate collection (clean rebuild for demo)
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass  # Collection didn't exist yet

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}  # Cosine similarity for retrieval
)
print(f"  Collection '{COLLECTION_NAME}' created (cosine similarity)\n")

# --- Embed and store chunks ---
print("Embedding and indexing chunks...")
texts = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]
ids = [f"chunk_{i:04d}" for i in range(len(chunks))]

# Process in batches to manage memory
BATCH_SIZE = 32
t0 = time.time()
for start in range(0, len(texts), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(texts))
    batch_texts = texts[start:end]

    # Generate embeddings for this batch
    batch_embeddings = embedder.encode(batch_texts).tolist()

    # Store in ChromaDB
    collection.add(
        documents=batch_texts,
        embeddings=batch_embeddings,
        metadatas=metadatas[start:end],
        ids=ids[start:end],
    )

indexing_time = time.time() - t0
print(f"  Indexed {collection.count()} chunks in {indexing_time:.1f}s")
print(f"  Average: {indexing_time/len(texts)*1000:.0f}ms per chunk")
print(f"\n  Knowledge base ready.")

## 4. Retrieval System — Semantic Search

This section defines query-time semantic retrieval over the vector store. It implements the Retrieval component by returning top-k context chunks most relevant to the question.

In [ ]:
# ============================================================
# Retrieval Function
# ============================================================
# The retrieval system takes a natural language query, embeds it,
# and finds the most semantically similar chunks in the knowledge base.

def retrieve(query: str, top_k: int = TOP_K) -> List[dict]:
    """
    Retrieve the top-K most relevant chunks for a given query.

    Process:
      1. Embed the query using the same model used for documents
      2. Search ChromaDB for nearest neighbors by cosine similarity
      3. Return results with text, metadata, and distance score

    Args:
        query: Natural language question from the user
        top_k: Number of chunks to retrieve (default: 5)

    Returns:
        List of dicts, each containing:
          - text: The chunk content
          - source: The source document filename
          - distance: Cosine distance (0 = identical, 2 = opposite)
    """
    # Embed the query into the same 384-dim space as the documents
    query_embedding = embedder.encode([query]).tolist()[0]

    # Search ChromaDB — returns results sorted by distance (ascending)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    # Package results into a clean format
    retrieved = []
    for i in range(len(results["documents"][0])):
        retrieved.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i].get("source", "unknown"),
            "distance": results["distances"][0][i],
        })

    return retrieved


# --- Demo: Test retrieval ---
print("RETRIEVAL DEMO")
print("=" * 60)

demo_query = "What are the benefits of edge computing?"
print(f"Query: \"{demo_query}\"\n")

results = retrieve(demo_query)
for i, r in enumerate(results, 1):
    print(f"  Result {i} | distance: {r['distance']:.4f} | source: {os.path.basename(r['source'])}")
    print(f"    {r['text'][:150]}...\n")

## 5. Augmentation — Context-Aware Prompt Construction

This section formats retrieved evidence and the user question into a grounded prompt template. It implements the Augmentation component that constrains generation to document-supported context.

In [ ]:
# ============================================================
# Augmentation Function
# ============================================================

def build_augmented_prompt(query: str, retrieved_chunks: List[dict]) -> list:
    """
    AUGMENTATION STEP: Combine retrieved context with user query.

    This function constructs the prompt that will be sent to the LLM.
    The prompt design is critical — it determines whether the LLM will:
      - Stay grounded in the retrieved context (vs. hallucinating)
      - Cite its sources (vs. making unsupported claims)
      - Admit when it doesn't have enough information (vs. fabricating)

    We use the chat template format expected by SmolLM2-Instruct:
      - system message: sets the behavior and rules
      - user message: contains the context + question

    Args:
        query: The user's original question
        retrieved_chunks: List of retrieved chunk dicts from the retrieve() function

    Returns:
        List of message dicts formatted for the model's chat template
    """
    # Format each retrieved chunk with a source label
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        source_name = os.path.basename(chunk["source"])
        context_parts.append(f"[Source {i}: {source_name}]\n{chunk['text']}")

    # Join chunks with clear separators
    context_block = "\n\n---\n\n".join(context_parts)

    # Build the messages in chat format
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant that answers questions based on provided context. "
                "Rules: (1) Use ONLY the information in the context below to answer. "
                "(2) Cite your sources by referring to [Source N]. "
                "(3) If the context does not contain enough information, say so clearly. "
                "(4) Keep your answer concise and direct."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context_block}\n\nQuestion: {query}",
        },
    ]

    return messages


print("build_augmented_prompt() function defined.")

## 6. Generation — Local Language Model

This section initializes the local instruct model used to generate final answers from augmented prompts. It implements the Generation component of RAG with CPU-only inference.

In [ ]:
# ============================================================
# Load the Generative Model (SmolLM2-1.7B-Instruct)
# ============================================================
# This downloads ~3.4 GB on first run. Subsequent runs load from cache.
# The model runs on CPU — no GPU required.

print("Loading generative model (this may take 1-2 minutes on first run)...")
print(f"  Model: {LLM_MODEL_NAME}")
print(f"  This is a 1.7B parameter model — small enough for CPU inference.\n")

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    dtype=torch.float32,  # float32 for CPU compatibility
    device_map="cpu",
)

load_time = time.time() - t0
print(f"  Model loaded in {load_time:.1f}s")
print(f"  Parameters: ~1.7B")
print(f"  License: Apache 2.0")
print(f"  Device: CPU")
print(f"\n  Generation model ready.")

In [ ]:
def generate(messages: list) -> dict:
    """
    GENERATION STEP: Send augmented prompt to the LLM and get a response.

    Uses the model's chat template to properly format the prompt,
    then generates a response with controlled parameters:
      - temperature=0.3: More deterministic, factual responses
      - top_p=0.9: Nucleus sampling for natural language
      - max_new_tokens=300: Enough for a thorough answer, not a novel

    Args:
        messages: List of message dicts (from build_augmented_prompt)

    Returns:
        Dict with 'answer', 'generation_time', and 'tokens_generated'
    """
    # Apply the model's chat template to format the prompt correctly
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Tokenize — using tokenizer() instead of encode() to get attention_mask
    inputs = tokenizer(prompt_text, return_tensors="pt")
    input_length = inputs["input_ids"].shape[1]

    # Generate
    t0 = time.time()
    with torch.no_grad():  # No gradient computation needed for inference
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,  # Avoid padding warnings
        )
    generation_time = time.time() - t0

    # Decode only the new tokens (exclude the prompt)
    new_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return {
        "answer": answer.strip(),
        "generation_time": generation_time,
        "tokens_generated": len(new_tokens),
        "tokens_per_second": len(new_tokens) / generation_time if generation_time > 0 else 0,
    }


print("generate() function defined.")

## 7. Full RAG Pipeline — End to End

This section composes retrieval, augmentation, and generation into a single callable pipeline. It is the integrated execution layer that operationalizes the complete RAG system.

In [ ]:
# ============================================================
# Complete RAG Pipeline
# ============================================================
# This function ties all five components together into one call.

def rag_query(query: str, top_k: int = TOP_K, verbose: bool = True) -> dict:
    """
    Full RAG pipeline: Query → Retrieve → Augment → Generate → Answer

    This is the main entry point. Given a natural language question,
    it returns a grounded, cited answer drawn from the knowledge base.

    Args:
        query: Natural language question from the user
        top_k: Number of chunks to retrieve (default: 5)
        verbose: Whether to print results (default: True)

    Returns:
        Dict containing:
          - query: The original question
          - answer: The LLM's grounded response
          - sources: List of retrieved chunks with metadata
          - timing: Breakdown of time spent in each stage
    """
    pipeline_start = time.time()

    # --- STEP 1: RETRIEVE ---
    t0 = time.time()
    retrieved_chunks = retrieve(query, top_k)
    retrieval_time = time.time() - t0

    # --- STEP 2: AUGMENT ---
    t0 = time.time()
    messages = build_augmented_prompt(query, retrieved_chunks)
    augmentation_time = time.time() - t0

    # --- STEP 3: GENERATE ---
    t0 = time.time()
    gen_result = generate(messages)
    generation_time = time.time() - t0

    total_time = time.time() - pipeline_start

    # Package the result
    result = {
        "query": query,
        "answer": gen_result["answer"],
        "sources": [
            {
                "source": os.path.basename(r["source"]),
                "distance": round(r["distance"], 4),
                "preview": r["text"][:120] + "...",
            }
            for r in retrieved_chunks
        ],
        "timing": {
            "retrieval_s": round(retrieval_time, 3),
            "augmentation_s": round(augmentation_time, 3),
            "generation_s": round(generation_time, 3),
            "total_s": round(total_time, 3),
        },
        "generation_stats": {
            "tokens_generated": gen_result["tokens_generated"],
            "tokens_per_second": round(gen_result["tokens_per_second"], 1),
        },
    }

    # --- DISPLAY ---
    if verbose:
        print(f"QUERY: {query}")
        print("=" * 60)
        print(f"\nANSWER:\n{result['answer']}")
        print(f"\n{'─' * 60}")
        print(f"SOURCES ({len(result['sources'])} retrieved):")
        for i, src in enumerate(result["sources"], 1):
            print(f"  [{i}] {src['source']}  (distance: {src['distance']})")
        print(f"\nTIMING:")
        print(f"  Retrieval:    {result['timing']['retrieval_s']:.3f}s")
        print(f"  Augmentation: {result['timing']['augmentation_s']:.3f}s")
        print(f"  Generation:   {result['timing']['generation_s']:.3f}s")
        print(f"  Total:        {result['timing']['total_s']:.3f}s")
        print(f"  Speed:        {result['generation_stats']['tokens_per_second']} tok/s")

    return result


print("rag_query() function ready — full pipeline in a single call.")

## 8. System Summary & Performance Metrics

This section reports end-to-end system characteristics and observed runtime metrics from the demo runs. It summarizes all five RAG components and their practical CPU performance.

In [ ]:
# ============================================================
# System Summary & Metrics
# ============================================================

print("=" * 60)
print("  SYSTEM SUMMARY")
print("=" * 60)

print(f"""
KNOWLEDGE BASE
  Store:           ChromaDB (persistent, cosine similarity)
  Indexed chunks:  {collection.count()}
  Chunk size:      {CHUNK_SIZE} chars, {CHUNK_OVERLAP} char overlap
  Documents:       {len(glob.glob(os.path.join(DATA_DIR, '*')))} files in {DATA_DIR}/

SEMANTIC LAYER
  Model:           {EMBEDDING_MODEL_NAME}
  Dimensions:      {embedder.get_embedding_dimension()}
  Size:            ~80 MB

RETRIEVAL
  Method:          Cosine similarity (top-{TOP_K})
  Typical speed:   <100ms per query

AUGMENTATION
  Strategy:        System prompt + labeled context + user query
  Citation:        [Source N] format
  Grounding:       "Answer ONLY from context" instruction

GENERATION
  Model:           {LLM_MODEL_NAME}
  Parameters:      ~1.7B
  License:         Apache 2.0
  Max new tokens:  {MAX_NEW_TOKENS}
  Temperature:     {TEMPERATURE}
  Device:          CPU (no GPU required)
""")

# Timing summary from demo queries
if results:
    avg_total = sum(r["timing"]["total_s"] for r in results) / len(results)
    avg_retrieval = sum(r["timing"]["retrieval_s"] for r in results) / len(results)
    avg_generation = sum(r["timing"]["generation_s"] for r in results) / len(results)
    print(f"AVERAGE TIMING ({len(results)} queries)")
    print(f"  Retrieval:  {avg_retrieval:.3f}s")
    print(f"  Generation: {avg_generation:.1f}s")
    print(f"  Total:      {avg_total:.1f}s")

## 9. Interactive Demonstration

This section executes representative questions through the full RAG workflow and shows grounded answers with sources. It demonstrates all components working together in a realistic question-answering flow.

In [ ]:
# ============================================================
# Demo — Run Example Queries
# ============================================================
# These queries demonstrate different aspects of the RAG system:
#   1. Direct factual question (should find a clear answer)
#   2. Comparative question (should synthesize from multiple sources)
#   3. Specific technical question (should retrieve targeted chunks)
#   4. Out-of-scope question (should admit lack of information)

demo_queries = [
    "What are the three main cloud computing service models?",
    "How do edge computing and cloud computing complement each other?",
    "What is Retrieval-Augmented Generation and how does it reduce hallucinations?",
    "What programming language is Python and who created it?",  # Out of scope
]

results = []
for i, query in enumerate(demo_queries, 1):
    print(f"\n{'#' * 70}")
    print(f"# DEMO QUERY {i}")
    print(f"{'#' * 70}\n")
    result = rag_query(query)
    results.append(result)
    print()

In [ ]:
# ============================================================
# Interactive Mode (for live demo during interview)
# ============================================================
# Run this cell during the live demo to take
# questions from interviewers in real-time.

def interactive_mode():
    """Interactive query loop for live demonstrations."""
    print("=" * 60)
    print("  INTERACTIVE RAG MODE")
    print("  Type a question and press Enter.")
    print("  Type 'quit' or 'exit' to exit.")
    print("=" * 60)

    while True:
        print()
        query = input("Your question: ").strip()
        if query.lower() in ("quit", "exit"):
            print("Exiting interactive mode.")
            break
        if not query:
            continue
        print()
        rag_query(query)

interactive_mode()

---
## Summary

This notebook demonstrated a complete, minimal RAG pipeline with five clearly
separated components:

1. **Knowledge Base** — Documents chunked and stored in ChromaDB
2. **Semantic Layer** — all-MiniLM-L6-v2 encodes text into 384-dim vectors
3. **Retrieval** — Cosine similarity search returns the top-K relevant chunks
4. **Augmentation** — Structured prompt combines context + query with grounding rules
5. **Generation** — SmolLM2-1.7B-Instruct produces a cited, grounded answer

Everything runs locally on CPU. No external APIs, no GPU, no Docker, no Ollama.
Install dependencies via `pip install -r requirements.txt` and run this notebook.

### What I would add for production
- Hybrid retrieval (vector + BM25 keyword search)
- Cross-encoder reranking for higher retrieval precision
- Streaming generation for better UX
- Automated evaluation with labeled test sets
- Guardrails for input validation and output safety